<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session04-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 4: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Does local alignment ever beat global on the same pair?

Align $x = \texttt{ATCG}$ against $y = \texttt{TCG}$, with match $= +1$, mismatch $= -1$ and
gap $= -1$. Global alignment (Needleman-Wunsch) has to align every residue of both sequences; local
alignment (Smith-Waterman) may start and end anywhere. Which one scores higher, and what are the two
scores?

**Fix and run.** The cell below fills both dynamic-programming matrices and prints the scores. One
line, marked `FIX`, fills the first column of the global matrix wrongly. Run the cell once as it
is: what do the two scores say? Then fix the line and run it again.

In [1]:
import numpy as np

def score(a, b, match=1, mismatch=-1):
    return match if a == b else mismatch

def global_score(x, y, gap=-1):
    S = np.zeros((len(x) + 1, len(y) + 1))
    for i in range(1, len(x) + 1):
        S[i, 0] = 0                  # FIX: aligning x[:i] against nothing costs i gaps
    for j in range(1, len(y) + 1):
        S[0, j] = j * gap
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            S[i, j] = max(S[i-1, j-1] + score(x[i-1], y[j-1]),
                          S[i-1, j] + gap, S[i, j-1] + gap)
    return S

def local_score(x, y, gap=-1):
    S = np.zeros((len(x) + 1, len(y) + 1))   # first row and column stay 0
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            S[i, j] = max(S[i-1, j-1] + score(x[i-1], y[j-1]),
                          S[i-1, j] + gap, S[i, j-1] + gap, 0)
    return S

x, y = "ATCG", "TCG"
G, L = global_score(x, y), local_score(x, y)
print("global matrix:\n", G)
print("global score:", G[-1, -1])
print("\nlocal matrix:\n", L)
print("local score: ", L.max())

global matrix:
 [[ 0. -1. -2. -3.]
 [ 0. -1. -2. -3.]
 [ 0.  1.  0. -1.]
 [ 0.  0.  2.  1.]
 [ 0. -1.  1.  3.]]
global score: 3.0

local matrix:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 2. 1.]
 [0. 0. 1. 3.]]
local score:  3.0


**If you finish early.**

1. Trace back through the fixed global matrix by hand: which alignment gives the score?
2. Change the gap penalty to $-2$. Which score changes, and why does the other one stay the same?

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

The first column of the global matrix holds the cost of aligning the
first $i$ residues of $x$ against nothing: $i$ gaps, so `S[i, 0] = i * gap`. With zeros there, the
leading `A` of $x$ could be skipped for free, and the "global" score became 3, the same as local:
the bug quietly turned global alignment into a local-like one.

Fixed, global gives **2**: the alignment is `ATCG` against `-TCG`, three matches and one gap. Local
gives **3**: `TCG` against `TCG`, leaving out the `A` altogether. Local alignment is never lower than
global on the same pair, because it is free to leave out whatever costs points. With gap $= -2$,
global falls to 1 and local stays 3, since the best local alignment has no gaps. Which one to use is
a biological choice: global for sequences that are related along their whole length, local for a
shared domain in otherwise different proteins.

</details>